# ResNet50 — Garlic Disease Classification (Multi-Run Experiment)

| Item | Value |
|---|---|
| **Architecture** | ResNet50 (ImageNet pretrained, frozen base) |
| **Input size** | 380 × 380 × 3 |
| **Strategy** | 3-seed independent runs for statistical robustness |
| **Optimizer** | Adam + ExponentialDecay (lr=1e-5) |
| **Loss** | CategoricalCrossentropy (label_smoothing=0.15) |
| **Regularisation** | Dropout 0.5 + L2 1e-5 + Class-weight balancing |

---


In [ ]:
# ========== 1. IMPORTS ========== #
import os
import glob
import time
import random
import shutil
import tempfile
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              top_k_accuracy_score, roc_curve, auc,
                              cohen_kappa_score, matthews_corrcoef,
                              balanced_accuracy_score)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)


In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = "/kaggle/working/report_ResNet50_MultiRun"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (380, 380, 3)   # keep same resolution as EfficientNetB4 baseline

BATCH_SIZE  = 64
EPOCHS      = 30

# --- Multi-run settings ---
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

# --- Result storage ---
all_runs_results = []

print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Seeds      : {RANDOM_SEEDS}")
print(f"XLA JIT    : ON")


In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = tf.cast(img, tf.float32)
        img   = resnet_preprocess(img)          # ResNet50 preprocessing
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl  = _make_split('train', training=True)
    val_ds,   n_val,   _,           _          = _make_split('val',   training=False)
    test_ds,  n_test,  test_fnames, test_lbl   = _make_split('test',  training=False)

    cw     = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names       = class_names,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    return train_ds, val_ds, test_ds, meta


def build_model(input_shape, num_classes, steps_per_epoch):
    """Build ResNet50 with custom classification head."""
    base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = Model(inputs=base.input, outputs=out)

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=1e-5,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=CategoricalCrossentropy(label_smoothing=0.15),
        metrics=['accuracy'],
    )
    return model


print("Helper functions defined.")
print(f"  create_tf_datasets  — tf.data pipeline (GPU-optimised)")
print(f"  build_model         — ResNet50 + custom head")


In [ ]:
# ========== 6. MULTI-RUN TRAINING EXPERIMENT ========== #
for run_idx, seed in enumerate(RANDOM_SEEDS):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{len(RANDOM_SEEDS)}  —  seed={seed}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)

    steps_per_epoch = meta.n_train // BATCH_SIZE

    model = build_model(INPUT_SHAPE, meta.num_classes, steps_per_epoch)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'resnet50_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=meta.class_weight_dict,
        callbacks=callbacks,
    )

    # --- Learning curves ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, keys, title in zip(axes,
                                [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
                                ['Accuracy', 'Loss']):
        ax.plot(history.history[keys[0]], label='Train')
        ax.plot(history.history[keys[1]], label='Validation')
        ax.set_title(f'{title} — Run {run_idx+1} (seed={seed})')
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.close()

    # --- Evaluation ---
    best_model = load_model(os.path.join(RESULT_DIR, 'resnet50_best.keras'))
    pred_probs = best_model.predict(test_ds, verbose=1)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=class_names, digits=4))

    # --- Confusion matrix ---
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    test_acc = np.mean(y_pred_run == y_true_run)
    all_runs_results.append({
        'run':            run_idx + 1,
        'seed':           seed,
        'accuracy':       test_acc,
        'precision':      report['weighted avg']['precision'],
        'recall':         report['weighted avg']['recall'],
        'f1_score':       report['weighted avg']['f1-score'],
        'per_class_metrics': {
            c: {'precision': report[c]['precision'],
                'recall':    report[c]['recall'],
                'f1-score':  report[c]['f1-score']}
            for c in class_names
        },
        'result_dir':     RESULT_DIR,
        'history':        history.history,
        'y_true':         y_true_run,
        'y_pred':         y_pred_run,
        'pred_probs':     pred_probs,
        'class_names':    class_names,
        'test_filenames': meta.test_filenames,
        'n_train':        meta.n_train,
        'n_val':          meta.n_val,
        'n_test':         meta.n_test,
    })

    print(f"\n  Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}"
          f"  R={report['weighted avg']['recall']:.4f}"
          f"  F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()

print("\n" + "="*70)
print(" ALL TRAINING RUNS COMPLETED")
print("="*70)


---
## Section 2 — Results Aggregation & Scientific Reports

Aggregate metrics across all runs, generate LaTeX tables, CSV summaries, and visualizations for publication.


In [ ]:
# ========== AGGREGATE RESULTS FROM ALL RUNS ========== #
print("\n" + "="*80)
print("AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls    = [r['recall']    for r in all_runs_results]
f1_scores  = [r['f1_score']  for r in all_runs_results]

overall_stats = {
    'Accuracy':  {'mean': np.mean(accuracies), 'std': np.std(accuracies), 'values': accuracies},
    'Precision': {'mean': np.mean(precisions), 'std': np.std(precisions), 'values': precisions},
    'Recall':    {'mean': np.mean(recalls),    'std': np.std(recalls),    'values': recalls},
    'F1-Score':  {'mean': np.mean(f1_scores),  'std': np.std(f1_scores),  'values': f1_scores},
}

print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 80)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print(f"              Individual runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

class_names = list(all_runs_results[0]['per_class_metrics'].keys())

per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values), 'std': np.std(values), 'values': values}

print("\nStatistics calculated successfully!")


In [ ]:
# ========== CREATE SCIENTIFIC REPORT TABLE ========== #
overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean':   [overall_stats[m]['mean'] for m in ['Accuracy','Precision','Recall','F1-Score']],
    'Std':    [overall_stats[m]['std']  for m in ['Accuracy','Precision','Recall','F1-Score']],
    'Run 1':  [accuracies[0], precisions[0], recalls[0], f1_scores[0]],
    'Run 2':  [accuracies[1], precisions[1], recalls[1], f1_scores[1]],
    'Run 3':  [accuracies[2], precisions[2], recalls[2], f1_scores[2]],
})
overall_df['Mean ± Std'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}", axis=1)

print("OVERALL PERFORMANCE METRICS (3 RUNS)")
print("="*80)
print(overall_df[['Metric', 'Mean ± Std', 'Run 1', 'Run 2', 'Run 3']].to_string(index=False))
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, "overall_metrics_summary.csv"), index=False)

per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        per_class_rows.append({
            'Class': class_name, 'Metric': metric.capitalize(),
            'Mean': stats['mean'], 'Std': stats['std'],
            'Mean ± Std': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            'Run 1': stats['values'][0], 'Run 2': stats['values'][1], 'Run 3': stats['values'][2],
        })
per_class_df = pd.DataFrame(per_class_rows)

print("\nPER-CLASS PERFORMANCE METRICS (3 RUNS)")
print("="*80)
for class_name in class_names:
    class_data = per_class_df[per_class_df['Class'] == class_name]
    print(f"\n{class_name}:")
    print(class_data[['Metric', 'Mean ± Std']].to_string(index=False))

per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, "per_class_metrics_summary.csv"), index=False)
print("\nSummary tables saved to CSV files!")


In [ ]:
# ========== GENERATE LATEX TABLE FOR PAPER ========== #
latex_overall = r"""\begin{table}[h]
\centering
\caption{Overall Performance Metrics of ResNet50 (Mean ± Std over 3 runs)}
\label{tab:resnet50_overall}
\begin{tabular}{lcccc}
\hline
\textbf{Metric} & \textbf{Mean ± Std} & \textbf{Run 1} & \textbf{Run 2} & \textbf{Run 3} \\
\hline
"""
for _, row in overall_df.iterrows():
    latex_overall += f"{row['Metric']} & {row['Mean ± Std']} & {row['Run 1']:.4f} & {row['Run 2']:.4f} & {row['Run 3']:.4f} \\\\\n"
latex_overall += r"""\hline
\end{tabular}
\end{table}
"""

latex_per_class = r"""\begin{table}[h]
\centering
\caption{Per-Class Performance Metrics of ResNet50 (Mean ± Std over 3 runs)}
\label{tab:resnet50_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
"""
for class_name in class_names:
    prec = per_class_stats[class_name]['precision']
    rec  = per_class_stats[class_name]['recall']
    f1   = per_class_stats[class_name]['f1-score']
    latex_per_class += (f"{class_name} & {prec['mean']:.4f} ± {prec['std']:.4f} & "
                        f"{rec['mean']:.4f} ± {rec['std']:.4f} & "
                        f"{f1['mean']:.4f} ± {f1['std']:.4f} \\\\\n")
latex_per_class += r"""\hline
\end{tabular}
\end{table}
"""

print(latex_overall)
print(latex_per_class)

with open(os.path.join(BASE_RESULT_DIR, "latex_tables.tex"), "w") as f:
    f.write("% Overall Metrics Table\n"); f.write(latex_overall)
    f.write("\n\n% Per-Class Metrics Table\n"); f.write(latex_per_class)
print("LaTeX tables saved to 'latex_tables.tex'")


In [ ]:
# ========== VISUALIZATION OF RESULTS ACROSS RUNS ========== #
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
means = [overall_stats[m]['mean'] for m in metrics_list]
stds  = [overall_stats[m]['std']  for m in metrics_list]
x_pos = np.arange(len(metrics_list))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar plot with error bars
bars = axes[0].bar(x_pos, means, yerr=stds, capsize=10, alpha=0.8,
                   color='steelblue', edgecolor='black')
axes[0].set_xlabel('Metrics', fontweight='bold'); axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_title('ResNet50 Performance (Mean ± Std, 3 runs)', fontweight='bold')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(metrics_list)
axes[0].set_ylim([0, 1.05]); axes[0].grid(axis='y', alpha=0.3)
for i, (m, s) in enumerate(zip(means, stds)):
    axes[0].text(i, m + s + 0.02, f'{m:.4f}\n±{s:.4f}', ha='center', fontsize=8, fontweight='bold')

# Box plot
bp = axes[1].boxplot([accuracies, precisions, recalls, f1_scores],
                     labels=metrics_list, patch_artist=True, showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
for patch in bp['boxes']:
    patch.set_facecolor('lightblue'); patch.set_alpha(0.7)
axes[1].set_title('Distribution Across 3 Runs', fontweight='bold')
axes[1].set_ylim([0, 1.05]); axes[1].grid(axis='y', alpha=0.3)

# Per-class F1-Score
class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds  = [per_class_stats[c]['f1-score']['std']  for c in class_names]
axes[2].bar(np.arange(len(class_names)), class_f1_means, yerr=class_f1_stds,
            capsize=5, alpha=0.8, color='coral', edgecolor='black')
axes[2].set_title('Per-Class F1-Score (Mean ± Std)', fontweight='bold')
axes[2].set_xticks(np.arange(len(class_names)))
axes[2].set_xticklabels(class_names, rotation=45, ha='right')
axes[2].set_ylim([0, 1.05]); axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "metrics_visualization.png"), dpi=300, bbox_inches='tight')
plt.show()
print("Visualizations saved!")


In [ ]:
# ========== GENERATE COMPREHENSIVE SUMMARY REPORT ========== #
report_lines = []
report_lines.append("="*100)
report_lines.append("ResNet50 - MULTI-RUN EXPERIMENT REPORT")
report_lines.append("="*100)
report_lines.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Random Seeds: {RANDOM_SEEDS}")
report_lines.append(f"Number of Runs: {len(RANDOM_SEEDS)}")

report_lines.append("\n" + "="*100)
report_lines.append("OVERALL PERFORMANCE METRICS (Mean ± Std)")
report_lines.append("="*100)
report_lines.append(f"{'Metric':<20} {'Mean ± Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
report_lines.append("-"*100)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    stats = overall_stats[metric]
    report_lines.append(f"{metric:<20} {stats['mean']:.4f} ± {stats['std']:.4f}      "
                        + f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("\n" + "="*100)
report_lines.append("PER-CLASS PERFORMANCE METRICS (Mean ± Std)")
report_lines.append("="*100)
for class_name in class_names:
    report_lines.append(f"\nClass: {class_name}")
    report_lines.append("-"*100)
    report_lines.append(f"{'Metric':<20} {'Mean ± Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
    report_lines.append("-"*100)
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        report_lines.append(f"{metric.capitalize():<20} {stats['mean']:.4f} ± {stats['std']:.4f}      "
                            + f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("\n" + "="*100)
report_lines.append("STATISTICAL SUMMARY")
report_lines.append("="*100)
best_idx  = np.argmax(accuracies)
worst_idx = np.argmin(accuracies)
report_lines.append(f"\nBest Run : Run {best_idx+1} (seed={RANDOM_SEEDS[best_idx]})  Accuracy={accuracies[best_idx]:.4f}")
report_lines.append(f"Worst Run: Run {worst_idx+1} (seed={RANDOM_SEEDS[worst_idx]})  Accuracy={accuracies[worst_idx]:.4f}")
report_lines.append(f"\nCoefficient of Variation (lower = more stable):")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    cv = (overall_stats[metric]['std'] / overall_stats[metric]['mean']) * 100
    report_lines.append(f"  {metric}: {cv:.2f}%")

report_lines.append("\n" + "="*100)
report_lines.append("END OF REPORT")
report_lines.append("="*100)

report_text = "\n".join(report_lines)
print(report_text)

with open(os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)
print("\nSummary report saved to 'MULTI_RUN_SUMMARY_REPORT.txt'")


In [ ]:
# ========== ZIP ALL RESULTS (Multi-Run) ========== #
import shutil

zip_output_path = "/kaggle/working/ResNet50_MultiRun_Complete"
shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)
zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"Archive size: {zip_size:.2f} MB")
print(f"Location    : {zip_output_path}.zip")
print("ALL DONE!")


---
## Section 3 — Advanced Analysis for Thesis / Paper

Reports below aggregate data from **all runs** (dataset, confusion matrix, ROC curves, Kappa/MCC) and then provide **single-run deep-dive** analysis (Grad-CAM, t-SNE, error analysis).

> **To analyse a specific run:** change `SELECTED_RUN` in the *Select Run* cell below (1 / 2 / 3).


In [ ]:
# ========== DATASET DISTRIBUTION ANALYSIS ========== #
splits = ['train', 'val', 'test']
split_counts = {}
for split in splits:
    split_dir = os.path.join(DATA_DIR, split)
    counts = {cls: len(os.listdir(os.path.join(split_dir, cls)))
              for cls in sorted(os.listdir(split_dir))
              if os.path.isdir(os.path.join(split_dir, cls))}
    split_counts[split] = counts

dist_df = pd.DataFrame(split_counts).fillna(0).astype(int)
dist_df['Total'] = dist_df.sum(axis=1)

print("Dataset Distribution:")
print(dist_df.to_string())
print(f"\nTotal images: {dist_df['Total'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dist_df[splits].plot(kind='bar', ax=axes[0],
                     color=['steelblue', 'coral', 'seagreen'],
                     edgecolor='black', width=0.7)
axes[0].set_title('Sample Count per Class per Split', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Number of Images')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(title='Split'); axes[0].grid(axis='y', alpha=0.3)
for container in axes[0].containers:
    axes[0].bar_label(container, fontsize=7, padding=1)

axes[1].pie(dist_df['Total'], labels=dist_df.index, autopct='%1.1f%%',
            startangle=140, colors=plt.cm.Set3.colors[:len(dist_df)])
axes[1].set_title('Overall Class Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
dist_df.to_csv(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.csv'))
print("Saved → dataset_distribution.png, dataset_distribution.csv")


In [ ]:
# ========== NORMALIZED CONFUSION MATRIX (Aggregate across runs) ========== #
cls    = all_runs_results[0]['class_names']
n_cls  = len(cls)
agg_cm = np.zeros((n_cls, n_cls))
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred']).astype(float)
agg_cm_norm = agg_cm / agg_cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(agg_cm.astype(int), annot=True, fmt='d',
            xticklabels=cls, yticklabels=cls, cmap='Blues', ax=axes[0])
axes[0].set_title('Aggregate Confusion Matrix (sum, 3 runs)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(agg_cm_norm, annot=True, fmt='.2%',
            xticklabels=cls, yticklabels=cls, cmap='Blues', ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix (row = recall)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.suptitle('Confusion Matrices — ResNet50 (Aggregate over 3 runs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print("\nPer-class recall (diagonal of normalized CM):")
for i, cname in enumerate(cls):
    print(f"  {cname:<25} {agg_cm_norm[i, i]:.4f}")
print("Saved → aggregate_confusion_matrix.png")


In [ ]:
# ========== MULTI-CLASS ROC CURVES + AUC (One-vs-Rest, Aggregate) ========== #
all_y_true = np.concatenate([r['y_true']    for r in all_runs_results])
all_probs  = np.concatenate([r['pred_probs'] for r in all_runs_results])
cls        = all_runs_results[0]['class_names']
n_cls      = len(cls)
y_bin      = label_binarize(all_y_true, classes=range(n_cls))

fig, axes   = plt.subplots(1, 2, figsize=(14, 6))
tab_colors  = plt.cm.tab10.colors
auc_scores  = {}
fpr_d, tpr_d = {}, {}

for i, cname in enumerate(cls):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr_d[i], tpr_d[i])
    auc_scores[cname] = roc_auc
    axes[0].plot(fpr_d[i], tpr_d[i], color=tab_colors[i], lw=2,
                 label=f'{cname}  (AUC={roc_auc:.4f})')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set(xlim=[0,1], ylim=[0,1.01],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='Per-Class ROC Curves (OvR) — Aggregate')
axes[0].legend(loc='lower right', fontsize=8); axes[0].grid(alpha=0.3)

all_fpr  = np.unique(np.concatenate([fpr_d[i] for i in range(n_cls)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_cls):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr /= n_cls
macro_auc = auc(all_fpr, mean_tpr)

axes[1].plot(all_fpr, mean_tpr, 'b-', lw=2, label=f'Macro-avg  (AUC={macro_auc:.4f})')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set(xlim=[0,1], ylim=[0,1.01],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='Macro-Average ROC Curve')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle('ROC Curves — ResNet50 (Aggregate over all runs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print("\nAUC Scores (aggregate):")
for cname, av in auc_scores.items():
    print(f"  {cname:<25} {av:.4f}")
weights      = [np.sum(all_y_true == i) for i in range(n_cls)]
weighted_auc = np.average(list(auc_scores.values()), weights=weights)
print(f"\n  Macro-average AUC   : {macro_auc:.4f}")
print(f"  Weighted-avg AUC    : {weighted_auc:.4f}")

auc_df = pd.DataFrame({'Class': list(auc_scores.keys()), 'AUC': list(auc_scores.values())})
auc_df = pd.concat([auc_df,
                    pd.DataFrame([{'Class': 'Macro-Average', 'AUC': macro_auc},
                                  {'Class': 'Weighted-Average', 'AUC': weighted_auc}])],
                   ignore_index=True)
auc_df.to_csv(os.path.join(BASE_RESULT_DIR, 'auc_scores.csv'), index=False)
print("Saved → roc_curves.png, auc_scores.csv")


In [ ]:
# ========== ADDITIONAL CLASSIFICATION METRICS (Kappa, MCC, Balanced Acc) ========== #
print("="*70); print("ADDITIONAL METRICS — ALL RUNS"); print("="*70)

extra_rows = []
for r in all_runs_results:
    kappa   = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc     = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    extra_rows.append({'Run': r['run'], 'Seed': r['seed'],
                       'Accuracy': r['accuracy'], 'Balanced Accuracy': bal_acc,
                       "Cohen's Kappa": kappa, 'MCC': mcc, 'F1-Score (w)': r['f1_score']})
    print(f"\nRun {r['run']} (seed={r['seed']}):")
    print(f"  Accuracy          : {r['accuracy']:.4f}")
    print(f"  Balanced Accuracy : {bal_acc:.4f}")
    print(f"  Cohen's Kappa     : {kappa:.4f}")
    print(f"  MCC               : {mcc:.4f}")
    print(f"  F1-Score (w-avg)  : {r['f1_score']:.4f}")

extra_df = pd.DataFrame(extra_rows)
num_cols = ['Accuracy', 'Balanced Accuracy', "Cohen's Kappa", 'MCC', 'F1-Score (w)']
mean_row = {'Run': 'Mean', 'Seed': '—', **{c: extra_df[c].mean() for c in num_cols}}
std_row  = {'Run': 'Std',  'Seed': '—', **{c: extra_df[c].std()  for c in num_cols}}
summary_extra = pd.concat([extra_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print("\n" + "="*70); print("SUMMARY TABLE")
print(summary_extra.to_string(index=False))
extra_df.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics.csv'), index=False)
summary_extra.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics_summary.csv'), index=False)
print("\nSaved → additional_metrics.csv, additional_metrics_summary.csv")


In [ ]:
# ========== TRAINING CONVERGENCE ANALYSIS ========== #
colors3 = ['steelblue', 'coral', 'seagreen']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for r, c in zip(all_runs_results, colors3):
    axes[0, 0].plot(r['history']['val_accuracy'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 0].set_title('Validation Accuracy — All Runs', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

for r, c in zip(all_runs_results, colors3):
    axes[0, 1].plot(r['history']['val_loss'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 1].set_title('Validation Loss — All Runs', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

best_epochs = [np.argmin(r['history']['val_loss']) + 1 for r in all_runs_results]
run_labels  = [f"Run {r['run']}\n(seed={r['seed']})" for r in all_runs_results]
bars = axes[1, 0].bar(run_labels, best_epochs, color=colors3[:len(all_runs_results)],
                      edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Best Epoch per Run (EarlyStopping)', fontweight='bold')
axes[1, 0].set_ylabel('Epoch'); axes[1, 0].grid(axis='y', alpha=0.3)
for bar, ep in zip(bars, best_epochs):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    str(ep), ha='center', fontweight='bold')

for r, c in zip(all_runs_results, colors3):
    gap = np.array(r['history']['accuracy']) - np.array(r['history']['val_accuracy'])
    axes[1, 1].plot(gap, color=c, label=f"Run {r['run']}", lw=2)
axes[1, 1].axhline(0, color='black', linestyle='--', lw=1)
axes[1, 1].set_title('Train−Val Accuracy Gap (Overfitting Indicator)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Train Acc − Val Acc')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle('Training Convergence Analysis — ResNet50',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'convergence_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → convergence_analysis.png")


In [ ]:
# ========== SELECT RUN FOR SINGLE-RUN ANALYSIS ========== #
SELECTED_RUN = len(RANDOM_SEEDS)   # default: last run

run_data    = all_runs_results[SELECTED_RUN - 1]
RESULT_DIR  = run_data['result_dir']
y_true      = run_data['y_true']
y_pred      = run_data['y_pred']
pred_probs  = run_data['pred_probs']
class_names = run_data['class_names']
test_acc    = run_data['accuracy']

n_train     = run_data['n_train']
n_val       = run_data['n_val']
n_test      = run_data['n_test']

test_filenames = run_data['test_filenames']
test_dir       = os.path.join(DATA_DIR, 'test')

history = SimpleNamespace(history=run_data['history'])

model = load_model(os.path.join(RESULT_DIR, 'resnet50_best.keras'))

_, _, test_ds, _ = create_tf_datasets(
    DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=run_data['seed'])

print(f"Analysing Run {SELECTED_RUN}  (seed={run_data['seed']})")
print(f"  Classes  : {class_names}")
print(f"  Train / Val / Test : {n_train} / {n_val} / {n_test}")
print(f"  Accuracy : {run_data['accuracy']:.4f}")
print(f"  F1-Score : {run_data['f1_score']:.4f}")
print(f"  Dir      : {RESULT_DIR}")


In [ ]:
# ========== GRAD-CAM VISUALIZATION ========== #

def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads  = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(orig_img_array, heatmap, alpha=0.4):
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet_colors    = plt.cm.jet(np.arange(256))[:, :3]
    jet_heatmap   = jet_colors[heatmap_uint8]
    jet_heatmap   = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap   = jet_heatmap.resize((orig_img_array.shape[1], orig_img_array.shape[0]))
    jet_heatmap   = img_to_array(jet_heatmap)
    superimposed  = jet_heatmap * alpha + orig_img_array
    return np.clip(superimposed / superimposed.max(), 0, 1)

last_conv_name = [l.name for l in model.layers
                  if isinstance(l, tf.keras.layers.Conv2D)][-1]
print(f"Last conv layer: {last_conv_name}")

grad_model = Model(inputs=model.inputs,
                   outputs=[model.get_layer(last_conv_name).output, model.output])

n_cls = len(class_names)
fig, axes = plt.subplots(n_cls, 3, figsize=(12, 4 * n_cls))
if n_cls == 1:
    axes = axes[np.newaxis, :]

for ci, cname in enumerate(class_names):
    ok_idx = np.where((y_true == ci) & (y_pred == ci))[0]
    idx    = ok_idx[0] if len(ok_idx) > 0 else np.where(y_true == ci)[0][0]
    fpath  = os.path.join(test_dir, test_filenames[idx])

    img_orig = load_img(fpath, target_size=INPUT_SHAPE[:2])
    img_arr  = img_to_array(img_orig)
    img_proc = resnet_preprocess(np.expand_dims(img_arr.copy(), 0))

    heatmap = make_gradcam_heatmap(img_proc, grad_model, pred_index=ci)
    overlay = overlay_heatmap(img_arr, heatmap)

    axes[ci, 0].imshow(img_orig); axes[ci, 0].axis('off')
    axes[ci, 0].set_title(f'Original\nClass: {cname}', fontsize=9)
    axes[ci, 1].imshow(heatmap, cmap='jet'); axes[ci, 1].axis('off')
    axes[ci, 1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[ci, 2].imshow(overlay); axes[ci, 2].axis('off')
    conf = pred_probs[idx][y_pred[idx]]
    axes[ci, 2].set_title(f'Overlay\nConf={conf:.2%}', fontsize=9)

plt.suptitle(f'Grad-CAM — ResNet50 (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → gradcam_visualization.png")


In [ ]:
# ========== t-SNE FEATURE EMBEDDING VISUALIZATION ========== #
gap_layer_name = [l.name for l in model.layers if 'global_average_pooling' in l.name][0]
feature_extractor = Model(inputs=model.input,
                          outputs=model.get_layer(gap_layer_name).output)

print("Extracting features from test set...")
features = feature_extractor.predict(test_ds, verbose=1)

print("Computing t-SNE embedding...")
tsne        = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(features)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette   = plt.cm.tab10.colors

for ci, cname in enumerate(class_names):
    mask = y_true == ci
    axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1],
                    c=[palette[ci]], label=cname, alpha=0.7, s=20)
axes[0].set_title(f't-SNE — True Labels (Run {SELECTED_RUN})', fontweight='bold')
axes[0].legend(markerscale=2, fontsize=9); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Dim 1'); axes[0].set_ylabel('Dim 2')

cm_mask = y_true == y_pred
axes[1].scatter(features_2d[cm_mask, 0], features_2d[cm_mask, 1],
                c='steelblue', label=f'Correct ({cm_mask.sum()})', alpha=0.6, s=20)
axes[1].scatter(features_2d[~cm_mask, 0], features_2d[~cm_mask, 1],
                c='red', label=f'Misclassified ({(~cm_mask).sum()})',
                alpha=0.9, s=50, marker='x')
axes[1].set_title('t-SNE — Correct vs Misclassified', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Dim 1'); axes[1].set_ylabel('Dim 2')

plt.suptitle(f't-SNE Feature Embedding — ResNet50 (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'tsne_embedding.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → tsne_embedding.png")


In [ ]:
# ========== ERROR ANALYSIS ========== #
print("="*70); print("ERROR ANALYSIS"); print("="*70)

confusion_counts = defaultdict(int)
for yt, yp in zip(y_true, y_pred):
    if yt != yp:
        confusion_counts[(class_names[yt], class_names[yp])] += 1

print(f"\nTotal misclassifications: {sum(confusion_counts.values())} / {len(y_true)}")
print(f"Overall accuracy: {np.mean(y_true == y_pred):.4f}\n")
print("Most common confused pairs (True → Predicted):")
for (tc, pc), cnt in sorted(confusion_counts.items(), key=lambda x: -x[1])[:10]:
    pct = cnt / np.sum(y_true == class_names.index(tc)) * 100
    print(f"  {tc:<22} → {pc:<22}  {cnt:3d} samples  ({pct:.1f}% of class)")

correct_mask = y_true == y_pred
correct_conf = pred_probs[correct_mask][np.arange(correct_mask.sum()), y_pred[correct_mask]]
wrong_conf   = pred_probs[~correct_mask][np.arange((~correct_mask).sum()), y_pred[~correct_mask]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correct_conf, bins=20, color='steelblue', alpha=0.7, edgecolor='black',
             label=f'Correct (n={len(correct_conf)})')
axes[0].hist(wrong_conf, bins=20, color='salmon', alpha=0.7, edgecolor='black',
             label=f'Wrong (n={len(wrong_conf)})')
axes[0].axvline(0.5, color='black', linestyle='--', lw=1)
axes[0].set_title('Prediction Confidence Distribution', fontweight='bold')
axes[0].set_xlabel('Confidence (softmax probability)'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(alpha=0.3)

class_acc = [(cn, np.mean(y_pred[y_true == ci] == ci), (y_true == ci).sum())
             for ci, cn in enumerate(class_names)]
class_acc.sort(key=lambda x: x[1])
names, accs, counts = zip(*class_acc)
bars = axes[1].barh(names, accs, color=plt.cm.RdYlGn(np.array(accs)),
                    edgecolor='black', height=0.6)
axes[1].set_title('Per-Class Accuracy (sorted)', fontweight='bold')
axes[1].set_xlabel('Accuracy'); axes[1].set_xlim([0, 1.15])
axes[1].grid(axis='x', alpha=0.3)
for bar, acc, n in zip(bars, accs, counts):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{acc:.3f}  (n={n})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'error_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → error_analysis.png")


In [ ]:
# ========== QUALITATIVE ANALYSIS — TOP-5 MISCLASSIFIED PER PAIR ========== #
TOP_N         = 5
GRADCAM_ALPHA = 0.45

last_conv_name = [l.name for l in model.layers
                  if isinstance(l, tf.keras.layers.Conv2D)][-1]
grad_model_vis = Model(inputs=model.inputs,
                       outputs=[model.get_layer(last_conv_name).output, model.output])

def _gradcam(img_path, target_class):
    img_orig = load_img(img_path, target_size=INPUT_SHAPE[:2])
    img_arr  = img_to_array(img_orig)
    img_proc = resnet_preprocess(np.expand_dims(img_arr.copy(), 0))
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model_vis(img_proc)
        class_score = preds[:, target_class]
    grads   = tape.gradient(class_score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()
    h_uint8    = np.uint8(255 * heatmap)
    jet_colors = plt.cm.jet(np.arange(256))[:, :3]
    jet_hm     = jet_colors[h_uint8]
    jet_hm     = tf.keras.preprocessing.image.array_to_img(jet_hm)
    jet_hm     = jet_hm.resize((img_arr.shape[1], img_arr.shape[0]))
    jet_hm     = img_to_array(jet_hm)
    superimposed = jet_hm * GRADCAM_ALPHA + img_arr
    superimposed = np.clip(superimposed / superimposed.max(), 0, 1)
    confidence   = float(preds.numpy()[0, target_class])
    orig_float   = np.clip(img_arr / 255.0, 0, 1)
    return orig_float, heatmap, superimposed, confidence

wrong_indices = np.where(y_true != y_pred)[0]
pair_dict = defaultdict(list)
for idx in wrong_indices:
    pair = (int(y_true[idx]), int(y_pred[idx]))
    pair_dict[pair].append((idx, float(pred_probs[idx][y_pred[idx]])))

analysis_dir = os.path.join(RESULT_DIR, "qualitative_analysis")
os.makedirs(analysis_dir, exist_ok=True)

for (ti, pi), samples in sorted(pair_dict.items(), key=lambda x: -len(x[1])):
    true_name  = class_names[ti]
    pred_name  = class_names[pi]
    top_samples = sorted(samples, key=lambda x: -x[1])[:TOP_N]
    n_show      = len(top_samples)
    fig, axes   = plt.subplots(n_show, 3, figsize=(12, max(3.5, n_show * 3.5)))
    if n_show == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f'True: {true_name}  →  Predicted: {pred_name}  ({len(samples)} wrong)',
                 fontsize=13, fontweight='bold', color='crimson', y=1.01)
    for col, title in enumerate(['Original Image', 'Grad-CAM Heatmap', 'Overlay (model focus)']):
        axes[0, col].set_title(title, fontsize=10, fontweight='bold', pad=6)
    for row, (idx, _) in enumerate(top_samples):
        fpath    = os.path.join(test_dir, test_filenames[idx])
        true_conf = float(pred_probs[idx][ti])
        pred_conf = float(pred_probs[idx][pi])
        try:
            orig, heatmap, overlay, _ = _gradcam(fpath, pi)
        except Exception as e:
            print(f"  [WARN] {e}"); continue
        top3_str = '\n'.join([f"  {class_names[k]}: {pred_probs[idx][k]:.2%}"
                               for k in np.argsort(-pred_probs[idx])[:3]])
        axes[row, 0].imshow(orig); axes[row, 0].axis('off')
        axes[row, 0].text(0.02, 0.02, f'True: {true_name}\n(p={true_conf:.2%})',
                          transform=axes[row, 0].transAxes, fontsize=8, color='lime',
                          fontweight='bold', bbox=dict(facecolor='black', alpha=0.55, pad=2))
        axes[row, 1].imshow(heatmap, cmap='jet'); axes[row, 1].axis('off')
        axes[row, 2].imshow(overlay); axes[row, 2].axis('off')
        axes[row, 2].text(0.02, 0.02, f'WRONG: {pred_name}\n(p={pred_conf:.2%})\n\nTop-3:\n{top3_str}',
                          transform=axes[row, 2].transAxes, fontsize=7.5, color='yellow',
                          fontweight='bold', bbox=dict(facecolor='black', alpha=0.60, pad=3))
    plt.tight_layout()
    safe_fname = f"top{TOP_N}_wrong_{true_name}_as_{pred_name}.png"
    plt.savefig(os.path.join(RESULT_DIR, safe_fname), dpi=200, bbox_inches='tight')
    plt.show()
    print(f"  [{true_name} → {pred_name}]  {len(samples)} errors — saved {safe_fname}")

print("\nQualitative analysis complete.")


---
## Section 4 — Single-Run Model Analysis & Full Report

Detailed model diagnostics: architecture, size, inference speed, Top-K accuracy, per-class metrics, and full summary report.


In [ ]:
# ========== MODEL SUMMARY & PARAMETERS ========== #
model.summary()

total_params      = model.count_params()
trainable_params  = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print("\n" + "="*60)
print(f"Total params        : {total_params:,}")
print(f"Trainable params    : {trainable_params:,}")
print(f"Non-trainable params: {non_trainable_params:,}")
print("="*60)

with open(os.path.join(RESULT_DIR, "model_summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
    f.write(f"\nTotal params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")
    f.write(f"Non-trainable params: {non_trainable_params:,}\n")
print("Model summary saved.")


In [ ]:
# ========== MODEL SIZE ========== #
temp_model_path = os.path.join(tempfile.gettempdir(), "temp_resnet50.keras")
model.save(temp_model_path)
model_size_bytes = os.path.getsize(temp_model_path)
model_size_mb    = model_size_bytes / (1024 * 1024)
os.remove(temp_model_path)

print(f"Model size: {model_size_mb:.2f} MB  ({model_size_bytes:,} bytes)")
with open(os.path.join(RESULT_DIR, "model_size.txt"), "w") as f:
    f.write(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)\n")

# ========== INFERENCE SPEED ========== #
print("\nWarming up...")
warmup_x = next(iter(test_ds.take(1)))[0][:8]
for _ in range(3):
    _ = model(warmup_x, training=False)

print("Measuring inference speed...")
MAX_BATCHES  = 10
total_images = 0; total_time = 0.0

for i, (batch_x, _) in enumerate(test_ds):
    if i >= MAX_BATCHES: break
    start_time = time.perf_counter()
    out = model(batch_x, training=False)
    _ = tf.reduce_sum(out).numpy()
    total_time   += time.perf_counter() - start_time
    total_images += len(batch_x)

avg_time_per_image = (total_time / total_images) * 1000
fps                = total_images / total_time

print(f"\nFPS          : {fps:.2f}")
print(f"ms / image   : {avg_time_per_image:.2f}")
print(f"Total images : {total_images}  in {total_time:.3f} s")

with open(os.path.join(RESULT_DIR, "inference_speed.txt"), "w") as f:
    f.write(f"FPS      : {fps:.2f}\n")
    f.write(f"ms/image : {avg_time_per_image:.2f}\n")
    f.write(f"Total    : {total_images} imgs in {total_time:.3f} s\n")

# ========== TOP-K ACCURACY ========== #
top_1_acc = top_k_accuracy_score(y_true, pred_probs, k=1, labels=range(len(class_names)))
top_3_acc = top_k_accuracy_score(y_true, pred_probs, k=3, labels=range(len(class_names)))
print(f"\nTop-1 Accuracy: {top_1_acc:.4f}")
print(f"Top-3 Accuracy: {top_3_acc:.4f}")

with open(os.path.join(RESULT_DIR, "topk_accuracy.txt"), "w") as f:
    f.write(f"Top-1 Accuracy: {top_1_acc:.4f}\nTop-3 Accuracy: {top_3_acc:.4f}\n")


In [ ]:
# ========== COMPREHENSIVE SUMMARY REPORT ========== #
summary_report = [
    "="*80,
    "ResNet50 - COMPREHENSIVE EVALUATION REPORT",
    "="*80,
    f"\nDataset: {DATA_DIR}",
    f"Result Directory: {RESULT_DIR}",
    f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "\n" + "-"*80, "MODEL CONFIGURATION", "-"*80,
    f"Architecture: ResNet50",
    f"Input Shape: {INPUT_SHAPE}",
    f"Number of Classes: {len(class_names)}",
    f"Classes: {', '.join(class_names)}",
    f"Total Parameters    : {total_params:,}",
    f"Trainable Parameters: {trainable_params:,}",
    f"Non-trainable Params: {non_trainable_params:,}",
    f"Model Size          : {model_size_mb:.2f} MB",
    "\n" + "-"*80, "DATASET STATISTICS", "-"*80,
    f"Training Samples  : {n_train}",
    f"Validation Samples: {n_val}",
    f"Test Samples      : {n_test}",
    "\n" + "-"*80, "TRAINING CONFIGURATION", "-"*80,
    f"Batch Size     : {BATCH_SIZE}",
    f"Epochs (actual): {len(history.history['loss'])}",
    f"Initial LR     : 1e-5",
    f"Optimizer      : Adam + ExponentialDecay",
    f"Loss           : CategoricalCrossentropy (label_smoothing=0.15)",
    f"Class Weights  : Balanced",
    "\n" + "-"*80, "PERFORMANCE METRICS", "-"*80,
    f"Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)",
    f"Top-1 Accuracy: {top_1_acc:.4f}",
    f"Top-3 Accuracy: {top_3_acc:.4f}",
    "\n" + "-"*80, "INFERENCE SPEED", "-"*80,
    f"FPS      : {fps:.2f}",
    f"ms/image : {avg_time_per_image:.2f}",
    "\n" + "="*80, "END OF REPORT", "="*80,
]
report_text = "\n".join(summary_report)
print(report_text)

with open(os.path.join(RESULT_DIR, "SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)
print("\nComprehensive summary report saved.")

# ========== ZIP SINGLE-RUN RESULTS ========== #
zip_output_path = "/kaggle/working/ResNet50_Complete_Report"
shutil.make_archive(zip_output_path, 'zip', RESULT_DIR)
zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\nArchive: {zip_output_path}.zip  ({zip_size:.2f} MB)")
print("ALL DONE!")
